# g1_limpo v2 — resume da `bloco8` na Kaggle

Continua a run `bloco8` do checkpoint `model_1050.pt`.

## Antes de rodar — quatro coisas

1. **Settings → Accelerator → GPU** (T4 ×1 ou P100).
2. **Settings → Internet → On.** O clone do GitHub precisa dela.
3. **Add Input → o dataset `G1-Limpo-V2`**, que contém `model_1050.pt`.
4. **A branch tem de estar no GitHub.** Na sua máquina:
   `git push -u origin exp/g1-limpo-v2`

## O que muda em relação ao notebook do Colab

| | Colab | Kaggle |
|---|---|---|
| persistência | Drive montado | `/kaggle/working`, e **só se a versão for commitada** |
| checkpoint | pasta gravável no Drive | dataset read-only e **plano** |
| GPU | desconhecida (23 022 passos/s a 8192 envs) | T4 16 GB (medido: 19 307 a 4096) |
| sessão | desconexão | **12 h duras** |

Três consertos vêm disso:

- **A semeadura da árvore de log.** O `get_checkpoint_path` resolve
  `<log_root>/g1_limpo/<run_dir>/model_N.pt`; o dataset é plano. A célula 4 copia o
  checkpoint para uma pasta de run com data 1900, que ordena abaixo de qualquer pasta
  real que o mjlab crie depois.
- **`NUM_ENVS` cai de 8192 para 4096.** Não cabe 8192 em 16 GB. O lote do PPO cai pela
  metade, e isso está impresso no log da célula 5.
- **`max_iterations` é limitado pelo relógio**, não só pela `META`. Uma versão
  commitada que passa de 12 h falha e não salva a saída.

## Para persistir

Rode com **Save Version → Save & Run All (Commit)**. Sessão interativa que expira leva
`/kaggle/working` embora — foi assim que o bloco 4 se perdeu.

Não existe célula de treino do zero neste notebook, de propósito: um commit roda tudo
de cima a baixo, e uma célula de `resume=False` apagaria o progresso em silêncio.

In [ ]:
# ⚠ NADA DE `import torch` NESTA CÉLULA. O torch registra operadores C++ no import,
# e se ele entrar no kernel ANTES do pip, um reload depois levanta
# `Only a single TORCH_LIBRARY can be used to register the namespace triton`.
import subprocess, sys

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU: Settings -> Accelerator -> GPU"
print("python", sys.version.split()[0])

## Dependências

**Instale só o `mjlab`.** Ele declara a árvore inteira e já pina o que importa:
`mujoco-warp~=3.10.0,>=3.10.0.3`, `mujoco~=3.10.0`, `rsl-rl-lib==5.4.0`, `numpy<2.5`,
`tensorboard>=2.20.0`, `scipy>=1.15`. Repetir esses pins à mão não adiciona segurança —
adiciona modos de falha, porque uma única versão indisponível derruba a resolução toda.

⚠ **A versão é 1.5.3, não 1.5.1.** O `RslRlModelCfg` de 1.5.1 manda `cnn_cfg` e
`rnn_type` que o `MLPModel` do `rsl-rl-lib 5.4.0` rejeita. O
`g1_multitask/kaggle/requirements.txt` ainda pede 1.5.1 — ele está velho, não o copie.

⚠ **`torch` não entra.** A imagem da Kaggle traz um build casado com o CUDA da máquina;
deixar o pip trocá-lo é o jeito mais rápido de perder a GPU sem perceber. O `mjlab`
exige `torch>=2.7.0` e as imagens atuais satisfazem.

In [ ]:
import subprocess, sys

# ⚠⚠ LISTA DE ARGUMENTOS, NUNCA STRING DE SHELL. Este foi um defeito medido: com
# `!pip install ... scipy>=1.15 numpy<2.5` o shell lê `>` e `<` como REDIRECIONAMENTO,
# tenta abrir um arquivo chamado `2.5`, aborta com exit 2 — e o pip NUNCA RODA. Com
# `-q` e sem conferir o código de saída, a célula não reclama e o erro só aparece duas
# células depois, como `No module named 'mjlab'`.
cmd = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "mjlab==1.5.3"]
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
assert r.returncode == 0, (
    "o pip falhou. A causa mais comum é a internet DESLIGADA: "
    "Settings -> Internet -> On, e rode esta célula de novo.")

r = subprocess.run([sys.executable, "-m", "pip", "list"],
                   capture_output=True, text=True)
alvo = ("mjlab", "rsl-rl-lib", "mujoco", "mujoco-warp", "warp-lang", "torch",
        "tensordict", "tyro", "tensorboard", "numpy", "scipy")
for linha in r.stdout.splitlines():
    if linha.split(" ")[0].lower() in alvo:
        print(linha)

## O pip levou a GPU?

Se ele trocou o torch por um build sem CUDA, **tudo abaixo roda em CPU sem reclamar** e
a sessão vira 12 h de nada. A checagem roda num subprocesso, porque `reload(torch)` não
existe.

In [ ]:
import subprocess, sys

chk = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-600:])
assert " True " in f" {chk.stdout} ", (
    "o pip trocou o torch e a CUDA foi embora. Reinstale com --no-deps o que puxou "
    "torch, ou reinicie o kernel e rode da célula 1")

# só agora, e é a PRIMEIRA vez neste processo
import torch, mjlab, mujoco, warp
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"{torch.cuda.get_device_name(0)}")
print("mjlab", mjlab.__version__ if hasattr(mjlab, "__version__") else "?",
      "| mujoco", mujoco.__version__, "| warp", warp.config.version)

## O repo, e a semeadura da árvore de log

A célula abaixo faz três coisas, e a terceira é o conserto central do porte.

⚠ **Se você deu `git push` depois de abrir esta sessão, RE-RODE esta célula.** A Kaggle
já rodou código antigo sem avisar uma vez; o sinal foi um peso de knob velho no primeiro
print.

⚠ **A task é registrada por efeito colateral do `import g1_limpo`.** Editar o arquivo
não basta: o kernel usa a versão em cache. Num commit isso não acontece (kernel novo);
numa sessão interativa, reinicie o runtime e rode da célula 1.

In [ ]:
import importlib, os, pathlib, re, shutil, subprocess, sys

os.environ.setdefault("MUJOCO_GL", "egl")

RUN      = "bloco13"            # ⚠ NOME NOVO: a v3.3 tirou o `alcança ≡ 1` do BOTAR,
                                # portanto `staged` e `precise_ori` medem ali outra coisa
                                # que na bloco12, e reusar o nome misturaria duas tabelas
                                # num log e numa impressão digital.
                                # A pasta-semente é criada a partir do arquivo do dataset,
                                # portanto o checkpoint continua sendo achado.
# ⚠ O CHECKPOINT É ACHADO PELO MAIOR NÚMERO, e não por nome fixo. O nome fixo era um
# número a manter em sincronia a cada sessão, e ele já falhou: o run fecha em `model_N`
# ou `model_N+1` conforme o `learn()` salvar o final, e adivinhar custava uma rodada.
IT_MINIMA = 4000                # trava contra dataset VELHO: o .pt tem de passar disto
BRANCH   = "exp/g1-limpo-v2"

RAIZ     = pathlib.Path("/kaggle/working/g1")     # o clone, refeito a cada sessão
LOG_ROOT = pathlib.Path("/kaggle/working/logs")   # ⚠ FORA de RAIZ
raiz_exp = LOG_ROOT / "g1_limpo"                  # <log_root>/<experiment_name>

# ------------------------------------------------------------------- 1. o clone
# ⚠ EXIGE INTERNET LIGADA e a branch NO GITHUB.
if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)],
               check=True)
print("clone   =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                  capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` NÃO é higiene. O Python guarda um finder POR DIRETÓRIO em
# `sys.path_importer_cache`, e o finder de um diretório que não existia no momento da
# inserção fica cacheado como VAZIO — `import g1_limpo` falharia com `No module named`
# mesmo com o pacote em disco.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

# --------------------------------------- 2. ACHAR o checkpoint dentro do dataset
# ⚠ `rglob` e não caminho fixo: a Kaggle monta ora em `/kaggle/input/<slug>/`, ora em
# `/kaggle/input/datasets/<usuario>/<slug>/`.
entrada = pathlib.Path("/kaggle/input")
assert entrada.is_dir() and any(entrada.iterdir()), \
    "nenhum dataset no input: Add Input -> G1-Limpo-V2"
achados = sorted(entrada.rglob("model_*.pt"),
                 key=lambda q: int(re.search(r"model_(\d+)", q.stem).group(1)),
                 reverse=True)
assert achados, (
    "não achei nenhum `model_*.pt` em /kaggle/input. O que existe lá:\n  "
    + "\n  ".join(str(p) for p in sorted(entrada.rglob("*.pt"))[:20]))
origem = achados[0]
ARQ_CKPT = origem.name
it_dataset = int(re.search(r"model_(\d+)", origem.stem).group(1))
assert it_dataset >= IT_MINIMA, (
    f"o dataset traz {ARQ_CKPT} (iteração {it_dataset}), abaixo da IT_MINIMA de "
    f"{IT_MINIMA}: é versão VELHA. Suba a saída da última sessão.")
print(f"dataset = {origem}  ({origem.stat().st_size / 2**20:.1f} MB)")

# --------------------------- 3. SEMEAR a árvore que o mjlab sabe procurar
# ⚠ O `get_checkpoint_path` resolve `<log_root>/<experiment_name>/<run_dir>/<ckpt>`, e o
# dataset da Kaggle é PLANO e read-only. Sem esta semeadura o resume levanta
# `No run directories found in /kaggle/working/logs/g1_limpo`.
#
# ⚠ E a pasta começa com 1900 DE PROPÓSITO. O `get_checkpoint_path` ordena os nomes e
# pega o ÚLTIMO; com data no passado, qualquer pasta real que o mjlab criar nesta sessão
# (`2026-...`) ordena acima, e um segundo resume pega o checkpoint NOVO, não a semente.
semente = raiz_exp / f"1900-01-01_00-00-00_{RUN}"
semente.mkdir(parents=True, exist_ok=True)
alvo = semente / ARQ_CKPT
if not alvo.exists():
    shutil.copy2(origem, alvo)
print(f"semente = {alvo}")

# a impressão digital dos pesos, se você a subiu junto no dataset
for p in sorted(entrada.rglob(f"{RUN}.pesos.json")):
    destino = raiz_exp / p.name
    if not destino.exists():
        shutil.copy2(p, destino)
        print(f"digital = {destino}  (veio do dataset)")
    break

print(f"\nlog_root = {LOG_ROOT}")
for p in sorted(raiz_exp.rglob("*")):
    print("  ", p.relative_to(LOG_ROOT))

## O pacote de saída, e o download

⚠ **Você não está commitando a versão, portanto `/kaggle/working` MORRE com a sessão.**
Esta célula define `empacota()`, e a célula do resume a chama num `finally` — o pacote
nasce no fim normal do treino E se ele estourar ou se você interromper o kernel.

⚠⚠ **O que código nenhum salva é a sessão MORTA** (12 h, timeout de inatividade, aba
fechada). Aí nada roda, nem isto. Para não depender disso, veja a célula do fim: subir
para um Dataset pela API da Kaggle é a única persistência que funciona sem você estar
presente.

In [ ]:
# =====================================================================
#  EMPACOTA E BAIXA — rode DEPOIS do treino, ou chame no `finally` dele.
# =====================================================================
import os, pathlib, re, zipfile
from IPython.display import FileLink, HTML, display

RUN      = "bloco13"
LOG_ROOT = pathlib.Path("/kaggle/working/logs")
SAIDA    = pathlib.Path("/kaggle/working")
ULTIMO_CKPT = ULTIMA_RUN = ULTIMA_IT = None


def empacota(run=RUN, baixa=True):
    """Zipa o último checkpoint + tfevents + params e oferece o download.

    ⚠ UM ZIP SÓ, e não os arquivos soltos. Um bloco de 1500 iterações deixa ~30
    checkpoints; baixar um por um é o que faz a sessão expirar no meio. E o tfevents
    vai junto porque sem ele o `leitura.py` não tem o que ler.
    """
    raiz_exp = LOG_ROOT / "g1_limpo"
    if not raiz_exp.is_dir():
        print(f"nada em {raiz_exp} — o treino não chegou a escrever")
        return None

    # ⚠ A pasta de run REAL, nunca a semente `1900-...` que a célula do repo criou.
    runs = sorted(p for p in raiz_exp.iterdir()
                  if p.is_dir() and p.name.endswith(run)
                  and not p.name.startswith("1900"))
    if not runs:
        print(f"nenhuma pasta de run nova em {raiz_exp} — o treino não salvou nada")
        return None
    nova = runs[-1]

    cks = sorted(nova.glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    if not cks:
        print(f"nenhum checkpoint em {nova.name} — nem 50 iterações rodaram")
        return None
    ultimo = cks[-1]
    it = int(re.search(r"(\d+)", ultimo.name).group(1))

    pacote = SAIDA / f"{run}_it{it}.zip"
    with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ultimo, ultimo.name)
        for ev in sorted(nova.glob("events.out.tfevents*")):
            z.write(ev, ev.name)
        dig = raiz_exp / f"{run}.pesos.json"
        if dig.exists():
            z.write(dig, dig.name)
        for p in sorted((nova / "params").glob("*.yaml")):
            z.write(p, f"params/{p.name}")
        # a lista do que FICOU para trás, para você saber o que perdeu se a sessão morrer
        z.writestr("INVENTARIO.txt",
                   f"run: {nova.name}\nultimo: {ultimo.name}\n"
                   f"checkpoints na sessao: {len(cks)}\n"
                   + "\n".join(p.name for p in cks) + "\n")

    print(f"run       = {nova.name}")
    print(f"salvos    = {len(cks)} checkpoints ({cks[0].name} .. {ultimo.name})")
    print(f"pacote    = {pacote.name}  ({pacote.stat().st_size / 2**20:.1f} MB)")
    print(f"conteudo  = {zipfile.ZipFile(pacote).namelist()}")
    # ⚠ GLOBAIS, para a célula de upload não repetir a busca (e não divergir dela).
    global ULTIMO_CKPT, ULTIMA_RUN, ULTIMA_IT
    ULTIMO_CKPT, ULTIMA_RUN, ULTIMA_IT = ultimo, nova, it
    print(f"\nna proxima sessao: ARQ_CKPT = {ultimo.name!r}")

    if baixa:
        # ⚠ O href TEM de ser relativo. O `FileLink` monta o link a partir do cwd do
        # kernel, e na Kaggle o cwd e /kaggle/working; caminho absoluto gera href
        # quebrado que abre uma pagina de erro.
        os.chdir(SAIDA)
        display(FileLink(pacote.name))
        # ⚠ TENTATIVA de clique automatico. O output da Kaggle roda em iframe com
        # sandbox, e download iniciado por script pode ser BLOQUEADO sem mensagem.
        # O link acima e o caminho garantido — nao confie so nisto.
        display(HTML(
            f'<a id="bx{it}" href="{pacote.name}" download="{pacote.name}"></a>'
            f'<script>document.getElementById("bx{it}").click();</script>'
            f'<p><b>Se nada baixou</b>: clique no link acima, ou Data &rarr; Output '
            f'&rarr; <code>{pacote.name}</code>.</p>'))
    return pacote


# roda de graça se ainda não há nada: só imprime e sai
empacota()

## O resume

Idêntico ao do Colab, com três acréscimos que a Kaggle impõe: `NUM_ENVS`, o teto de
parede, e a impressão digital lida do dataset.

**Sobre a segunda T4.** Não está ligada, e o motivo é ARITMÉTICO, não medo. O tempo de
iteração é 89% física e 10% update, e os DOIS são data-parallel: com 4096 envs POR RANK
você ganha o LOTE de volta (196 608 transições, o da bloco8) com o mesmo tempo de
iteração — e **zero** iterações por hora a mais. Para ganhar iterações/hora seria preciso
2048 por rank, e aí cada T4 fica sub-utilizada e o ganho cai para ~1,5×.

Somando: um T4 com 4096 envs a ~7 s/iter fecha as 3950 iterações que faltam em **~7,7 h**,
dentro de uma sessão. A segunda GPU compra lote, não tempo. E cobra: o estado do
currículo (`env.limpo_forma`) é POR PROCESSO, portanto os dois ranks tocam rampas
independentes e só o rank 0 vai para o checkpoint; o seed do rank 1 é 43 e não 42; e o log
dos dois ranks inunda a célula. Se quiser mesmo, `cfg.gpu_ids = "all"` e `NUM_ENVS` é
**por rank**.

In [ ]:
# =====================================================================
#  RESUME — g1_limpo v2. O run vem de `RUN`, acima.  KAGGLE.
# =====================================================================
import dataclasses, json, pathlib, sys
import torch
sys.path.insert(0, str(RAIZ))

import g1_limpo
from g1_limpo import comando as CMD, observacoes as OB
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.velocity.config.g1.env_cfgs import unitree_g1_flat_env_cfg
from mjlab.utils.os import get_checkpoint_path

# ⚠ 8192 ERA O COLAB, NUM A100 DE 40 GB. Um T4 tem 16 GB e não segura isso. Ao cair
# para 4096 o LOTE do PPO cai pela metade (`num_envs × num_steps_per_env`), e com
# `num_mini_batches = 4` o minilote vai de 49 152 para 24 576 transições. É mudança de
# otimização no meio da run, e é o hospedeiro que a impõe — não uma escolha.
# Se der OOM: 2048. Numa máquina maior: 8192, igual ao Colab.
NUM_ENVS   = 4096
ENVS_ANTES = 4096
META       = 9500     # a iteração de DESTINO do bloco, ABSOLUTA
#            ⚠ 9500 = a bloco12 fecha na 7500, mais as 2000 deste bloco. Se o
#            resgate do 7500 falhar, ele retoma da 6999 e a META dá 2501.

# ⚠⚠ O TETO DE PAREDE DA KAGGLE É DURO: 12 h; e sem commit, a sessão morta leva
# `/kaggle/working`. O `SEG_POR_ITER` está ANCORADO NUMA MEDIÇÃO, e não num chute:
#
#   4096 envs, 1× T4, g1_multitask (2026-08-18): collection 4,558 s + learning 0,533 s
#                                                = 5,09 s/iter, 19 307 passos/s
#   8192 envs, a bloco8 no Colab:                 7,554 + 0,986 = 8,54 s/iter, 23 022
#
# ⚠ NÃO SEI QUAL GPU RODOU A bloco8. Os 23 022 passos/s com 8192 envs são só 1,19× os
# 19 307 de um T4 com 4096, portanto o "A100" era suposição minha — trate a linha da
# bloco8 como GPU desconhecida.
#
# 7,0 s deixa margem para a cena do g1_limpo ser mais caruda que a do multitask (mesa,
# caixa, 28 termos de recompensa). CORRIJA com o `Collection time` real do primeiro log.
HORAS_LIMITE = 10.5   # 12 h menos o pip, o clone e a montagem do env
SEG_POR_ITER = 7.0

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID),
                          log_root=str(LOG_ROOT))
cfg.env.scene.num_envs = NUM_ENVS
cfg.agent.run_name = RUN
cfg.agent.logger = "tensorboard"

# ------------------------------------------------------------------ O RESUME
cfg.agent.resume = True
# ⚠ `load_run` PINADO. O default `.*` casa QUALQUER run do experimento, e o
# `get_checkpoint_path` pega a de nome mais ALTO em ordem alfabética.
cfg.agent.load_run = f".*_{RUN}$"
cfg.agent.load_checkpoint = r"model_\d+\.pt"

# ⚠ A LR **NÃO** BAIXA AQUI. A regra dos 5e-4 vale para warm-start numa distribuição
# NOVA. Este resume devolve currículo, `common_step_counter` e o estado do Adam: a
# distribuição é a MESMA e a função de valor não está velha.
assert cfg.agent.algorithm.schedule == "adaptive"
assert cfg.agent.algorithm.learning_rate == 1.0e-3
# ⚠⚠ ISTO É WARM-START, E NÃO RESUME. A tabela de recompensa mudou na v2.1 (o
# `renda_congelada` acrescenta um nível que o crítico nunca viu), portanto a função de
# valor está errada por vários pontos por segundo. O degrau de LR é o que o próprio
# guarda de impressão digital manda fazer, e o que a nota `warmstart-lower-lr` registra.
# O transiente dura 120 a 150 iterações; não julgue o run antes de +200.
cfg.agent.algorithm.learning_rate = 5.0e-4
assert cfg.agent.seed == 42, \
    "trocar o seed troca a população de robôs no meio do treino"

# ------------------------------- ACHAR E LER O CHECKPOINT ANTES DE LANÇAR
ckpt = get_checkpoint_path(raiz_exp.resolve(), cfg.agent.load_run,
                           cfg.agent.load_checkpoint)
sd = torch.load(ckpt, map_location="cpu", weights_only=True)
it0 = int(sd["iter"])
print(f"checkpoint  = {ckpt}")
print(f"iteração    = {it0}")

# ⚠⚠ `max_iterations` É ADITIVO. `rsl_rl/runners/on_policy_runner.py:78` faz
#     total_it = current_learning_iteration + num_learning_iterations
# Com 5000 aqui e 1050 no checkpoint, o treino iria até 6050.
falta_meta = META - it0
assert falta_meta > 0, f"o checkpoint já passou da META: {it0} >= {META}"
cabe_tempo = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)
cfg.agent.max_iterations = min(falta_meta, cabe_tempo)
manda = "a META" if falta_meta <= cabe_tempo else "o RELÓGIO"
print(f"até a META  = {falta_meta} iterações")
print(f"cabe em {HORAS_LIMITE:.1f} h a {SEG_POR_ITER:.1f} s/iter = {cabe_tempo}")
print(f"vai rodar   = {cfg.agent.max_iterations}  (manda {manda}) "
      f"-> termina na {it0 + cfg.agent.max_iterations}")
if falta_meta > cabe_tempo:
    print(f"⚠ ESTA SESSÃO NÃO ALCANÇA A {META}. Suba o último checkpoint como nova "
          f"versão do dataset e rode este notebook de novo.")

# --- O CURRÍCULO VOLTOU? Sem ele a rampa recomeça no piso: ~400 iterações.
estado = (sd.get("infos") or {}).get("limpo_curriculo") or {}
assert "forma" in estado, (
    "checkpoint SEM estado de currículo: o `RunnerComEstadoDeCurriculo` não estava "
    "registrado quando ele foi salvo. A rampa voltaria a 0,95 e a carência de 200 "
    "iterações seria re-paga (ver `runner.py`)")
frm = estado["forma"]
print("currículo   = alvo {alvo:.3f}  razao {razao:.3f}  "
      "iters_balanco {iters_balanco:.0f}  abriu {abriu:.0f}".format(**frm))
# ⚠ `s_B`/`s_C` (spec §2.5): o checkpoint ANTIGO (pré dois-bits) não os tem — o
# `garante_forma` semeia o piso na falta deles. Não é um assert: travaria o warm-start.
if "s_B" in frm and "s_C" in frm:
    print(f"balanceador = s_B {frm['s_B']:.3f}  s_C {frm['s_C']:.3f}")
else:
    print("balanceador = ausente no checkpoint; será semeado no piso (s_B=0, s_C=1)")

# ⚠ A TRUNCAGEM DE ENVS É ESPERADA AQUI, e hoje é de graça: o `nivel` está em 0 em
# todos os envs, portanto perder a cauda não perde dificuldade nenhuma. No dia em que
# `Curriculum/nivel` sair de 0, cair de 8192 para 4096 joga metade fora.
for nome in ("limpo_nivel", "limpo_elo"):
    assert nome in estado, f"falta `{nome}` no estado do currículo"
    n = int(estado[nome].numel())
    if n != NUM_ENVS:
        print(f"⚠ {nome}: checkpoint com {n} envs, sessão com {NUM_ENVS}. O runner "
              f"copia o que cabe; o resto nasce no default (nível 0).")

# --- É UM CHECKPOINT DA v2? A dimensão da observação é o discriminador honesto.
def _n_entradas(sdict):
    for k, v in sdict.items():
        if k.endswith("obs_normalizer._mean"):
            return int(v.shape[-1])
    raise SystemExit(f"normalizador não encontrado; chaves: {sorted(sdict)[:6]}")

n_ator, n_critico = (_n_entradas(sd["actor_state_dict"]),
                     _n_entradas(sd["critic_state_dict"]))
print(f"observação  = ator {n_ator}, crítico {n_critico}")
assert (n_ator, n_critico) == (114, 131), (
    f"o checkpoint tem ator {n_ator} / crítico {n_critico}, e a v2 tem 114 / 131. "
    "Retomar isto enxertaria a entrada errada na rede, em silêncio")

# --- DERIVA DE CONFIG ENTRE SESSÕES. O `params/env.yaml` do mjlab NÃO volta a ser
# lido (tags `python/object` somem quando a classe muda de lugar), portanto a
# impressão digital é nossa.
#
# ⚠ NA KAGGLE ELA SÓ FUNCIONA SE VIAJAR NO DATASET. `/kaggle/working` nasce vazio a
# cada sessão, portanto suba o `bloco8.pesos.json` junto com o checkpoint na próxima
# versão do dataset — a célula anterior o copia de volta se ele estiver lá.
#
# ⚠ SÓ A LEITURA MORA AQUI. A gravação fica na última linha, depois de todos os
# asserts: gravar antes deixaria um clone antigo estampar os pesos velhos.
digital = {k: float(v.weight) for k, v in cfg.env.rewards.items()}
arq = raiz_exp / f"{RUN}.pesos.json"
if arq.exists():
    antigo = json.loads(arq.read_text())
    dif = {k: (antigo.get(k), digital.get(k))
           for k in set(antigo) | set(digital) if antigo.get(k) != digital.get(k)}
    assert not dif, (
        f"peso de recompensa mudou desde o último lançamento: {dif}. Se a mudança é "
        "intencional, isto é WARM-START e não resume: apague o json e baixe a LR "
        "para 5e-4.")
else:
    print(f"⚠ sem impressão digital anterior; gravando {arq.name} no fim")

# =====================================================================
#  DAQUI PARA BAIXO É IDÊNTICO À CÉLULA DE TREINO. O clone é NOVO em toda
#  sessão, portanto os asserts de "é a v2 mesmo?" valem igual.
# =====================================================================
rw, cu, tm, ev = (cfg.env.rewards, cfg.env.curriculum,
                  cfg.env.terminations, cfg.env.events)
fab = unitree_g1_flat_env_cfg(play=False)

# --- O CLONE É A v2? Estes cinco falham alto num clone antigo. ---
assert CMD.DIM == 12 and CMD.GIRO == slice(9, 12), \
    f"comando com DIM={CMD.DIM}: clone anterior ao canal `giro_b` (spec §8.3)"
assert OB.N_CAIXA == 10, f"N_CAIXA={OB.N_CAIXA}: clone sem `giro_b`/`meia_aresta`"
assert len(CMD.CADEIAS) == 3 and all(CMD.CARREGAR not in c for c in CMD.CADEIAS), \
    f"CADEIAS = {CMD.CADEIAS}: clone anterior ao dois-bits (CARREGAR virou CAUDA, spec §2.1)"
# ⚠ v2.1: o `load` SAIU e o `renda_congelada` entrou. O congelamento paga TODO fecho,
# portanto o fecho do BOTAR rende mais que pairar sem o `load` — e sem número escolhido
# à mão. Ver docs/planos/2026-09-04-proposta-gradientes-g1-limpo.md §3 P3.
assert "largou" not in rw and "load" in rw, \
    f"clone fora do dois-bits: load={'load' in rw}, largou={'largou' in rw}"
assert list(rw)[-1] == "renda_congelada", \
    f"`renda_congelada` tem de ser o ÚLTIMO termo (ele lê `_step_reward` dos outros); hoje o último é {list(rw)[-1]!r}"
assert "tamanho_caixa" in ev and ev["tamanho_caixa"].mode == "startup", \
    "sem o evento de startup a caixa tem tamanho fixo e a obs lê zero no `meia_aresta`"

# --- contrato do pacote ---
assert list(cfg.env.commands) == ["twist", "alvo_caixa"], list(cfg.env.commands)
assert cfg.env.commands["alvo_caixa"].resampling_time_range[0] > cfg.env.episode_length_s, \
    "resample dentro do episódio zera o sucesso um passo antes do time_out"
assert tm["time_out"].time_out is True, "o rsl_rl trataria time_out como fracasso"
assert list(cu) == ["command_vel", "forma", "nivel", "elo"], \
    f"ordem do currículo errada ({list(cu)}): `forma` e `nivel` leriam o elo do episódio SEGUINTE"
assert cfg.agent.experiment_name == g1_limpo.EXPERIMENT == "g1_limpo"

# --- a OBSERVAÇÃO da v2: ator 114, crítico 131 (114 + 12 de pé do fabricante + 5) ---
assert list(cfg.env.observations["actor"].terms)[-2:] == ["elo", "caixa"], \
    list(cfg.env.observations["actor"].terms)
assert list(cfg.env.observations["critic"].terms)[-3:] == ["elo", "caixa", "elo_interno"], \
    "sem `elo_interno` o crítico confunde a espera final com um env parado, e o " \
    "`PPOPorElo` agrupa a espera final na locomoção (spec §6.1)"

# --- PARIDADE: a locomoção é a do fabricante ---
# ⚠ 14 termos nossos: os 8 da tarefa, as 3 multas de mesa (viraram multa em 01/09, e
# ANTES eram terminação), o `largou` da v2, e o `renda_congelada` da v2.1. O `load` saiu.
assert set(rw) - set(fab.rewards) == {
    "staged", "precise_pos", "precise_ori", "squeeze", "unload",
    "postura_ereta", "load", "terminacao", "joint_acc",
    "contato_tronco", "contato_palma", "contato_dorso",
    "renda_congelada", "velocidade_por_regime"}, \
    f"termo inventado na locomoção: {sorted(set(rw) - set(fab.rewards))}"
assert not set(fab.rewards) - set(rw), "termo do fabricante desapareceu"
assert cfg.env.actions["joint_pos"].scale == fab.actions["joint_pos"].scale
assert cu["command_vel"].params["velocity_stages"] == \
    fab.curriculum["command_vel"].params["velocity_stages"]
assert rw["dof_pos_limits"].weight == -1.0
assert rw["action_rate_l2"].weight == fab.rewards["action_rate_l2"].weight == -0.1, \
    "o action_rate é o do fabricante; −1,0 é do g1_poc, que tem currículo nesse peso"
assert set(fab.events) - set(ev) == {"base_com"}, "dr.body_com_offset corrompe a heap"
assert set(ev) - set(fab.events) == {"carga_caixa", "posiciona_cena", "tamanho_caixa"}, \
    f"eventos nossos: {sorted(set(ev) - set(fab.events))}"

# --- OS DISCRIMINADORES DE CLONE ANTIGO ---
assert sorted(tm) == ["caixa_largada", "fell_over", "time_out"], \
    f"terminações {sorted(tm)} — as 3 multas de mesa viraram RECOMPENSA em 01/09; " \
    "com elas aqui, o clone é anterior a isso"
assert {"palmas_em_contato", "dorso_em_contato", "impacto_da_caixa"} <= set(cfg.env.metrics), \
    "sem `impacto_da_caixa` não há como ver a política começando a JOGAR a caixa"
assert "mu" in rw["squeeze"].params, \
    "o `squeeze` ainda usa `forca_ref` fixo de 12,0 N — clone anterior ao 917bf38"
assert "sensores_palma" in rw["unload"].params, \
    "o `unload` está sem porteiro: derrubar a caixa paga 2,0/s sem mão nenhuma"
# --- OS DISCRIMINADORES DA v2.1 (auditoria de gradientes, 2026-09-04) ---
assert rw["precise_pos"].weight == 3.0 and rw["precise_pos"].params["sigma"] == 0.18, \
    f"o preciso não cobre o aceite: peso {rw['precise_pos'].weight}, "\
    f"σ {rw['precise_pos'].params['sigma']} — com σ 0,05 ele paga 0,018 no limiar do fecho"
assert "nome_do_comando" in rw["load"].params and "sensor_apoio" in rw["load"].params, \
    "clone sem `load` completo: sem ele nada paga a caixa apoiada no BOTAR (spec §2.7)"
assert "elos_que_andam" not in rw["track_linear_velocity"].params, \
    "o rastreio ainda é gateado por CONJUNTO DE ELOS: as duas esperas e o segurar-parado "\
    "pagam 4,0/s por velocidade zero forçada (P4)"
assert "pesos_manip" in cu["elo"].params, \
    "o sorteio de elo é uniforme: o REORIENTAR inerte come metade da manipulação (P8)"
# --- OS DISCRIMINADORES DE 08/09 (o sigma da tarefa, e o robo lento) ---
assert "velocidade_por_regime" in rw and rw["velocidade_por_regime"].weight < 0, \
    "o limite de velocidade de junta esta ausente, ou na forma POSITIVA — a positiva "\
    "paga 2,0/s a um robo PARADO, em todo env (piso da estatua)"
assert rw["velocidade_por_regime"].params["vel_max_standing"] == {".*": 2.0}, \
    "o limite do `standing` mudou: 2,0 rad/s e o p99 MEDIDO da locomocao parada"
# ⚠ `nome_do_comando` SAIU dos params do rastreio (spec dois-bits §2.7): `engajado`
# passou a ser só `limpo_pegou`, sem `× VALIDA`, e o VALIDA era a única razão de
# `rastreio_por_elo` precisar do nome do comando. O discriminador agora é o WRAPPER.
assert rw["track_linear_velocity"].func.__name__ == "rastreio_por_elo" \
    and "func" in rw["track_linear_velocity"].params \
    and "nome_do_comando" not in rw["track_linear_velocity"].params, \
    "clone fora do dois-bits: o rastreio precisa do wrapper `rastreio_por_elo` SEM "\
    "`nome_do_comando` — senao ele nao paga nos elos parados com a caixa na mao"
assert rw["pose"].func.__name__ == "PosturaPorElo", \
    "sem `PosturaPorElo` o braço sai da média em todo elo, e o `pose` do molde "\
    "vale 0,000 com derivada ZERO a 10% da faixa de junta"
assert "velocidade_de_junta" in cfg.env.metrics, \
    "sem a metrica os tres dicts de limite so recalibram com sondagem de CPU"
assert hasattr(cfg.env.commands["alvo_caixa"], "reorientar_inerte"), "clone antigo"
assert not hasattr(cfg.env.commands["alvo_caixa"], "prob_por_nivel"), \
    "clone com `prob_por_nivel`: anterior ao balanceador B/C (spec §2.5)"
assert hasattr(cfg.env.commands["alvo_caixa"], "botar_delta_topo"), \
    "clone sem `botar_delta_topo`: anterior à abertura do BOTAR na base (spec §1.4)"
# --- OS DISCRIMINADORES DE v3.2 (soltar termina, spec `g1-limpo-soltar-termina.md` §7) ---
assert "v_solta" in tm["caixa_largada"].params \
    and "dist_max" not in tm["caixa_largada"].params, \
    "clone anterior ao v3.2: `caixa_largada` ainda usa `dist_max` (distância às "\
    "palmas) em vez de `v_solta` (velocidade relativa, spec §2)"
assert {"aproxima_caixa", "renda_manipulacao"} <= set(cfg.env.metrics) \
        and cfg.env.metrics["impacto_da_caixa"].reduce == "max", \
    "sem a régua da caixa não há como ler se ela se aproxima do alvo (P9, P10)"
# --- O DISCRIMINADOR DE v3.3 (a mão gateia o BOTAR, spec `g1-limpo-mao-no-alcancar.md` §5) ---
# ⚠ A IMPRESSÃO DIGITAL NÃO PEGA ESTA MUDANÇA: ela compara só `weight`, e nenhum peso
# muda. Este assert é a ÚNICA trava contra retomar num clone pré-v3.3.
import inspect
from g1_limpo import recompensas as RC
assert "ones_like" not in inspect.getsource(RC._alcancar), \
    "clone anterior à v3.3: o `_alcancar` ainda devolve 1 constante no BOTAR — "\
    "segurar a caixa paga 3,99/s sem exigir a mão (spec §0)"
assert not str(LOG_ROOT).startswith(str(RAIZ)), \
    "log dentro do clone: o re-clone apaga o checkpoint"

assert cfg.agent.logger == "tensorboard", \
    "o default do mjlab é wandb, e o `leitura.py` lê events.out.tfevents"

lote_antes = ENVS_ANTES * cfg.agent.num_steps_per_env
lote_agora = NUM_ENVS * cfg.agent.num_steps_per_env
print(f"\nrecompensas = {len(rw)} (esperado 28)")
print(f"CADEIAS     = {CMD.CADEIAS}   DIM = {CMD.DIM}   N_CAIXA = {OB.N_CAIXA}")
print(f"envs        = {NUM_ENVS} (antes {ENVS_ANTES})")
print(f"lote do PPO = {lote_agora} transições (antes {lote_antes}), "
      f"minilote {lote_agora // cfg.agent.algorithm.num_mini_batches}")
print(f"resume      = de {ckpt.name} (it {it0})")
print(f"log_root    = {cfg.log_root}")
print("\n⚠ O mjlab abre uma pasta de run NOVA, com o timestamp de agora. As iterações")
print("  continuam de onde pararam, mas o `events.out.tfevents` é OUTRO arquivo.")
print("\n[CONFERE] tudo ok. retomando.\n")

# ⚠ A GRAVAÇÃO DA IMPRESSÃO DIGITAL É A ÚLTIMA COISA ANTES DO LANÇAMENTO.
arq.write_text(json.dumps(digital, indent=1, sort_keys=True))

# ⚠ `try/finally`, e não a linha nua. Assim o pacote de saída nasce também quando o
# treino estoura (OOM é o caso provável num T4) ou quando você interrompe o kernel. O
# `empacota` vem da célula anterior.
try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    empacota()

## Persistência sem commit — subir o checkpoint pela API

⚠ **Isto é o que funciona sem você presente.** O download do navegador exige a aba
aberta e um clique; isto não exige nada. É a resposta certa para "não estou rodando com
commit".

Precisa de duas coisas, uma vez só:

1. **Add-ons → Secrets** → crie `KAGGLE_USERNAME` e `KAGGLE_KEY`. Os valores estão em
   <https://www.kaggle.com/settings> → API → **Create New Token** (baixa um
   `kaggle.json` com os dois campos).
2. **Internet → On.**

⚠ Ele sobe o **`.pt` cru**, e não o zip. É de propósito: a célula do repo procura
`ARQ_CKPT` com `rglob`, portanto um `.pt` no dataset é achado direto na próxima sessão.
Um zip obrigaria a descompactar antes.

⚠ **Uma versão nova SUBSTITUI o conteúdo do dataset.** O `model_1050.pt` sai. Isso é o
desejado — e a Kaggle guarda as versões antigas, dá para voltar a qualquer uma.

In [ ]:
# =====================================================================
#  SOBE O CHECKPOINT COMO NOVA VERSÃO DO DATASET  (não precisa de commit)
# =====================================================================
import json, os, pathlib, shutil, subprocess

SLUG = "g1-limpo-v2"        # o slug do SEU dataset, sem o nome de usuário

from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
usuario = os.environ["KAGGLE_USERNAME"]

# reusa a busca do `empacota` — sem repetir a lógica e sem chance de divergir dela
if ULTIMO_CKPT is None:
    empacota(baixa=False)
assert ULTIMO_CKPT is not None, "nada para subir: o treino não salvou checkpoint"

# ⚠ PASTA PRÓPRIA, com SÓ o que sobe. O `kaggle datasets version` envia o diretório
# INTEIRO — apontá-lo para /kaggle/working mandaria o clone do repo e os 30
# checkpoints junto.
envio = pathlib.Path("/kaggle/working/envio")
shutil.rmtree(envio, ignore_errors=True)
envio.mkdir()
shutil.copy2(ULTIMO_CKPT, envio / ULTIMO_CKPT.name)
dig = LOG_ROOT / "g1_limpo" / f"{RUN}.pesos.json"
if dig.exists():
    shutil.copy2(dig, envio / dig.name)
(envio / "dataset-metadata.json").write_text(json.dumps(
    {"title": "G1-Limpo-V2", "id": f"{usuario}/{SLUG}",
     "licenses": [{"name": "CC0-1.0"}]}, indent=1))
print("vai subir:", sorted(p.name for p in envio.iterdir()))

r = subprocess.run(["kaggle", "datasets", "version", "-p", str(envio),
                    "-m", f"{RUN} it{ULTIMA_IT}", "-r", "skip"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, (
    "o upload falhou. Confira: os dois segredos existem e estão ANEXADOS a este "
    "notebook (Add-ons -> Secrets -> o toggle de cada um), a internet está ligada, "
    f"e o dataset {usuario}/{SLUG} existe e é seu.")
print(f"\nsubiu {ULTIMO_CKPT.name} em {usuario}/{SLUG}")
print(f"na próxima sessão: ARQ_CKPT = {ULTIMO_CKPT.name!r}")

## A próxima sessão, em quatro passos

**Se você rodou a célula da API**, o dataset já está atualizado: troque `ARQ_CKPT` na
célula do repo pelo nome que ela imprimiu, e rode de novo. Nada mais.

**Se você só baixou o zip**, descompacte, suba o `model_*.pt` em `G1-Limpo-V2` → **New
Version**, e troque o `ARQ_CKPT`.

O `META = 5000` não muda entre sessões: ele é a iteração de destino absoluta, e a
célula do resume calcula sozinha quantas faltam.

**O que olhar no primeiro log que aparecer**, na ordem:

| canal | o que ele responde |
|---|---|
| `Metrics/alvo_caixa/sucesso` | saiu de 0? é a única linha que destrava o `nivel` |
| `Episode_Reward/unload` ÷ fatia de manipulação | ele continua erguendo a caixa? |
| `Metrics/twist/eficiencia_min` ÷ `Curriculum/elo` | a locomoção ficou acima de 0,60? |
| `Episode_Metrics/impacto_da_caixa` | passou de 6,2? é o piso, não o pico |
| `Collection time` | corrija o `SEG_POR_ITER` com o valor real |